The "rd" problem -- why naive pipelining breaks
   Here's a problem... By cycle 5, when the instruction wants to write back to
   register `rd`, that `rd` field was decoded from the instruction way back in 
   cycle 2. But the IF/ID register now contains a completely diffeent 
   instruction (the one that was fetched 4 cycles later). So the `rd` field
   coming from IF/ID is wrong -- it belongs to the newest instruction, not the 
   one trying to write back.

   The solution(page 22): PIPELINE THE `rd` FIELD TOO. The destination register 
   number gets carried forward through every pipeline register: 
      IF/ID -> ID/EX -> ... -> MEM/WB. 
   By the time the instruction reaches WB, the correct `rd` value has traveled
   with it through all the pipeline registers. It's like putting a shipping 
   label on a package -- the label follows the package through every station
   of the assembly line.


CONTROL SIGNALS GET PIPELINED TOO
   Same problem, same solution. The control signals are generated during ID 
   (decode), but some of them aren't needed until MEM (like MemRead, MemWrite)
   or WB (like RegWrite, MemToReg). So the control signals also get pushed
   through the pipeline registers, split into three groups:

   - EX signals (ALUSrcA, ALUSrcB, ALUOp) -- used in the next stage, stored in
     ID/EX.
   - MEM signals (MemRead, MemWrite, Branch) -- stored in ID/EX, then forwarded
     through EX/MEM.
   - WB signals (RegWrite, MemtoReg) -- stored in ID/EX --> forwarded through
     EX/MEM --> forwarded through MEM/WB.

   Each group travels exactly as far as it needs to. WB signals make the
   longest journey (3 pipeline registers) because they're not used until 3 
   stages after decode.


HAZARDS -- the complications (briefly)
   The slides mention three problem that pipelining introduces...:
      1. DATA HAZARDS: instruction needs a value that hasn't been computed yet
         (e.g., `add x1, x2, x3` followed by `sub x4, x1, x5` -- x1 isn't ready
         yet). Solutions: forwarding/bypassing, stalling.
      2. CONTROL HAZARDS: branches -- you don't know which instruction to fetch
         next until the branch is resolved. Solutions: branch prediction,
         delayed branches.
      3. EXCEPTIONS IN A PIPELINE: multiple instrucions in-flight means
         multiple things can go wrong simultaneously.

   These are "beyond this module" according to the slides, but they're the
   meat of any advanced architecture course.


---                        


#### WALKTHROUGH -- what to try

1. Basic Pipeline flow: hit ...
   Five instrutions flow through the 5-stage pipeline. The main grid shows each
   instruction as a row and each cycle advances them one stage to the right. 
   Watch how by cycle 4 (the 5th clock edge), all 5 stages are occupied 
   simultaneously -- that's a full pipeline. The colored cells show what each
   instruction is doing at each stage, with detail text explaining the
   actual operation. 

2. PIPELINE REGISTERS PANEL
   Below the main grid, you'll see the four pipeline registers (IF/ID, ... 
   MEM/WB) and their contents at the current cycle. Watch how `ld` starts in
   IF/ID at cycle 1, moves to ... The data literally flows rightward through 
   the registes each cycle.

3. TOGGLE "Show rd Hazard" (the red button)
   This is the key teaching moment... With this ON, every pipeline register
   shows the `rd` field travelling with this instruction (tagged in red:
   "rd=x1 travelling")   

   ... that's why we pipeline rd...

   Also notice at cycle 4: `sub` is in ID reading x1 and x2, but `ld` hasn't
   written x1 yet (it's just now in WB). That's a DATA HAZARD -- sub would 
   get the old `x1` value.

4. TOGGLE "Show Ctrl Signals"
   This shows the three control signal groups travelling pipeline registers:
      - EX signals (ALUSrc, ALUOp, RegDst) -- generated at ID, used at EX
        (trael through 1 pipeline reg).
      - MEM signals (MemRead, MemWrite, Branch) -- generated at ID, used at MEM
        (travel through 2 pipeline regs)
      - WB signals (RegWrite, MemToReg) -- generated at ID, used at WB (travel
        through 3 pipeline regs)

   Each group has a visual journey showing which stages it passes through, with
   "USED" highlighted at its destination. WB signals make the longest trip --
   they're carried through... before finally being consumed.

5. Timing Comparison (bottom)
   Sequential: 5 instructions \times 1000ps = 5000ps.
   Pipelined: 1000 ps to fill + 4 \times 200 ps = 1800ps.

   That's 2.8x faster -- and it gets closer to 5 x with more instructions
   (the stedy-state throughput is 1 instruction per 200ps instead of per 100ps).

   SPECIFIC CYCLES TO PAUSE AT:
      - Cycle 0: Only ld in IF. Pipeline nearly empty.
      - Cycle 4: Full pipeline! All 5 stages occupied by different instructions.
        This is peak utilisation.
      - Cycle 5: ld has completed (past WB), add is in WB, sub in MEM, sd in
        EX, ... New instruction just entered IF.
      - Cycle 8: Pipeline draining -- only addi left in wb.



THE HAZARDS PANEL at the very bottom gives. apreview... "beyond this module".       




---

---

### LECTURE 9, PART 3: Beyond -- Real-World Pipelines, SoCs, and Where it All Goes
   
BRISKI: a real RISC-V pipelined processor (p24)
   Everything we've been building towards -- the 5-stage pipeline -- the 5-stage
   pipeline with IF, ID, EX, MEM, WB -- isn't -- just a textbook abstraction.
   The slides show BRISKI, an actual RISC-V processor core with exactly this
   5-stage pipeline architecture, Looking at its datapath, you can spot all the
   pieces we've studies: the PC and instruction memory on the left (FETCH), the
   instruction decoder and register file (DECODE), the alu ...

   The pipeline registers sit between each stage as vertical bars, exactly where
   we placed them. There's also branch logic in the execute stage feeding back
   to the PC MUX -- that's the control hazard resolution we briefly mentioned.
   
   The point is: what we learned isn't simplified-beyond-recognition. A real
   RISC-V core looks almost exactly like our diagrams, just with a few more 
   wires for forwarding and branch handling.


---
#### Programmable System-on-Chip: where RISC processors live today (P 26-28)
   So far we've studied a "processor-on-chip" -- one CPU core on a piece of 
   silicon. But modern technology can fit much more than a single processor on
   a chip. The slides introduce PROGRAMMABLE SYSTEM-ON-CHIP (SoC), specifically
   the Xilinx Versal architecture (now AMD, since AMD acquired Xilinx).

   The Versal chip has three types of computation engines on a single piece of
   7nm silicon:

   SCALAR ENGINE: These are RISC processors -- specifically dual-core ARM
   Cortex-A72 (application processor) and dual-core ARM Cortex-R5 (real-time
   processor). The A72 is a high-performance core running Linux, managing the
   system. The R5 handles time-critical tasks. These are the direct descendants
   of everything we've studied -- pipelined RISC cores, just much more 
   sophisticated (out-of-order execution, branch prediction, caches, etc.)

   ADAPTABLE ENGINES: These are FPGA (Field-Programmable Gate Array) fabric --
   fine-grained programmable logic gates and registers. You can literally 
   configure these to implement any digital circuit. Want a custom accelerator
   for a specific algorithm? Program it into the adaptable engines. This is the 
   "field-programmable technology" ...

   AI Engines: These are vector optimised for AI and 5G workloads. Here's the
   mind-blowing part -- each AI engine itself contains a RISC scalar processor
   alongside the vector units. So even inside the AI accelerator, there's a tiny
   RISC core handling control flow while the vector units crunch the heavy math.
   RISC processors are everywhere, at every level.

   The Versal ... The key takeaway: the RISC architecture we've studied isn't
   just "one processor" -- it's a fundamental building block that appears
   multiple times within a single modern chip, from the main application cores
   down to the control processors inside AI accelerators.


APPLICATION-SPECIFIC Architecutes (page 29)
   ... rather than general-purpose processors, they design architectures 
   optimised for specific application domains:
      ...
      
   These all leverage the FPGA/adaptable engine technology described above. The
   core insight is: when you know your workload, you can build hardware that's
   orders of magnitude more efficient than a general-purpose CPU. But the 
   general-purpose cores are still there -- they handle the control plane, the 
   OS, the coordination between specialised engines. 


REVISION HINTS (p30)
   ... exam advice, worth taking seriously:

   - UNDERSTAND THE BIG PICTURES: datapath diagrams and state diagrams at 
     different levels of detail. If someone asks "draw the multi-cycle datapath",
     you should be able to sketch the key components, MUXes, and control
     signals from memory.
   - KEY EQUATIONS: execution time = IC \times CPI \times cycle time, Amdahl's
     Law (speedup = 1/ ((1 - f) + f/S)). Know how to apply these, not just 
     recite them.
   - READ QUESTIONS CAREFULLY: estimate marks per sub-question (e.g., 15% of#
     20 marks = 3 marks, so don't write a page for a 3-mark answer)...
   - ANSWER SELDOM EXCEED 2 PAGES: be concise and targeted
   - PRACTICE UNDER EXAM CONDITIONS: timed, without notes, pen and paper.
   
               

---